# betting-ad-blocker — the whole pipeline

## HOW TO USE

**The routine workflow is three steps. You never edit code.**

1. **Add a video** → drop the file into `input/videos/`
2. **Add a brand** → make a folder `input/logos/<brand>/` and drop logo images in it
3. **Run All** (Kernel ▸ Restart & Run All)

### Pinning the hide colour for a brand (recommended)

The panel that covers an ad should read as *that ad's* colour — Betano red for
Betano, blue for a blue brand. Drop a **`color.json`** next to the logos:

```
input/logos/betano/color.json     {"rgb": [164, 60, 51]}
input/logos/<other>/color.txt     #1E50C8          ← hex also works
```

With a colour pinned, that brand's panel uses exactly that colour, every time —
a half-occluded or motion-blurred frame can never produce a wrong one. Without
it, the colour is sampled from the LED strip's own face on the first clean
frame and frozen for the life of the track. Either way the colour is muted a
little (saturation/brightness down, **hue untouched**) so it doesn't glare.

Every step below detects what is already done and **skips it**. So a re-run
after adding one new video only extracts that video, only regenerates what
depends on it, and fine-tunes from the existing model rather than training
from scratch. Nothing outside `input/` and the config cell below needs to be
touched — no editing `scripts/`, no separate commands to remember.

**To redo a step on purpose**, set its `FORCE_*` flag in the config cell to
`True` (e.g. `FORCE_TRAIN = True` after changing training settings).

**To wipe everything generated and start clean**, set `RESET = True` in Step 0.
That deletes frames/synthetic/dataset/models — never anything in `input/`.

### What the steps do

| Step | Does | Skipped when |
|---|---|---|
| 0 | Optional reset | `RESET = False` (default) |
| 1 | Environment + GPU check | never (it's a check) |
| 2 | Extract frames from videos | that video already has frames |
| 3 | Auto-label real betting boards | those frames already auto-labelled |
| 4 | Generate synthetic training images | logo folders unchanged since last run |
| 5 | Assemble train/val dataset | never (cheap, always merges) |
| 6 | Auto-label people in the dataset | dataset unchanged since last run |
| 7 | Train (or fine-tune from `models/best.pt`) | `models/best.pt` exists |
| 8 | Sanity-check the model on held-out frames | never (it's a check) |
| 9 | Process a video → hidden-ad output | that output file already exists |

---
## Config

Everything tunable lives here. These values are written to `config.local.yaml`,
which overlays `config.yaml` — so `config.yaml` keeps its documentation and
defaults, and you can delete `config.local.yaml` any time to go back to them.

In [ ]:
# ── what to run ───────────────────────────────────────────────────────────
DEBUG            = True    # Step 9 also writes a side-by-side diagnostic video
PROCESS_VIDEO    = "match.mp4"   # filename inside input/videos/ (None = first found)
PROCESS_START    = None    # seconds; None = from the beginning
PROCESS_END      = None    # seconds; None = to the end

# ── force flags: set True to redo a step that would otherwise be skipped ──
FORCE_EXTRACT    = False
FORCE_AUTOLABEL  = False
FORCE_SYNTHETIC  = False
FORCE_PERSONS    = False
FORCE_TRAIN      = False
FORCE_PROCESS    = False

# ── frame extraction ──────────────────────────────────────────────────────
EXTRACT_FPS      = 1.0     # frames sampled per second of video

# ── training ──────────────────────────────────────────────────────────────
TRAIN_IMGSZ      = 1280    # NOT 640: thin perimeter boards vanish at 640
TRAIN_EPOCHS     = 60
FINETUNE_EPOCHS  = 30      # used automatically when models/best.pt exists

# ── detection / tracking (recall matters more than precision here) ────────
INFER_CONF       = 0.10    # feeds the TRACKER only, never the renderer
PANEL_CREATE_CONF   = 0.35 # a panel is born only from detections this strong…
PANEL_CREATE_FRAMES = 3    # …seen this many frames running
MERGE_GAP_HEIGHT_RATIO = 1.5  # merge same-line strips within this × height
PANEL_MARGIN_FRAC   = 0.02 # panel may never exceed detections + this margin

# ── fill colour: the ad's own colour, muted ───────────────────────────────
# Pin a brand's colour by dropping color.json into input/logos/<brand>/ - see
# "HOW TO USE" above. Unpinned brands sample their board instead.
SAMPLE_BAND_FRAC        = 0.6   # sample the middle 60% of the board's face
SATURATION_MULT         = 0.75  # take the edge off without washing the hue out
VALUE_MULT              = 0.8   # take off the shine
SCENE_BLEND             = 0.0   # leave at 0: blending hue toward the scene is
                                # what previously made the red board look brown
MAX_RELATIVE_BRIGHTNESS = 1.0   # never brighter than the scene average
BRAND_COLOR_SNAP_HUE    = 30    # sampled colour within this many hue degrees of
                                # a pinned brand snaps to it; farther away it is
                                # left alone (no cross-brand contamination)
PANEL_FEATHER_PX        = 4

# ── ghost-panel prevention (a panel may only exist where a board is) ──────
MAX_PROPAGATION_FRAMES  = 6     # frames a panel may coast with no detection
PAN_DROP_PX             = 25    # camera motion above this + no detection = the
                                # board is leaving frame, so drop it
VERIFY_CONTENT          = True  # re-check the pixels under a coasting panel
CONTENT_HUE_TOL         = 30
FIELD_OF_PLAY           = [0.15, 0.45, 0.85, 1.0]  # a board is never mid-pitch

# ── panel geometry guards ─────────────────────────────────────────────────
# Measured PERPENDICULAR to the strip, not as bounding-box height: a slanted
# perimeter board is ~85px thick but its bounding box spans ~270px, so a
# bbox-height cap would amputate its far end.
MAX_BOARD_HEIGHT_FRAC = 0.09   # ~146px at 1620p vs ~85-92px real strips
BOTTOM_SNAP           = True   # pin the panel's lower edge to the board/grass
                               # seam, killing the red sliver of un-hidden advert
BOTTOM_SNAP_MAX_FRAC  = 0.35   # max correction, as a fraction of thickness

# ── who may punch a hole in the panel ─────────────────────────────────────
# Only people IN FRONT of the board. The crowd behind it is hidden from the
# shins down, so their masks end at the board's top edge; someone in front is
# standing on grass and continues below its bottom edge.
OCCLUSION_ZONE_DOWN_FRAC = 0.25--
OCCLUSION_ZONE_UP_PX     = 6
FEET_TOLERANCE_FRAC      = 0.25
PROTECT_GOAL             = True  # never paint over goal posts / crossbar

# ── people (they always win over the panel) ───────────────────────────────
PERSON_CONF        = 0.10  # low on purpose: missing a person paints over them
PERSON_HOLD_FRAMES = 15    # keep a missed person's mask alive this long
PERSON_DILATE_PX   = 9
ROI_PERSON_PASS    = True  # second person pass inside each panel region
MOTION_PERSON_FALLBACK = True  # last-resort masks for figures BOTH detector
                           # passes miss: grass-anchored moving blobs. Kept
                           # conservative because an LED board's own content
                           # animates and would otherwise read as motion.
FRONT_LATCH_FRAMES = 2     # the in-front verdict LATCHES after this many frames
                           # and is never re-decided - re-deciding it per frame
                           # is what made the black-kit referee blink
ROI_UPSCALE        = 2.0   # the crop is UPSCALED before inference. The old
                           # fixed imgsz=640 ran it at 0.69x native - a
                           # downscale - which is why the black-kit referee
                           # kept blinking.

In [ ]:
import json, subprocess, sys, hashlib
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
PY = sys.executable

from common import load_config, find_videos, slugify, IMAGE_EXTS   # noqa: E402

# Write only the keys this notebook manages; config.yaml keeps its comments.
overlay = {
    "extract": {"fps": EXTRACT_FPS},
    "train": {"imgsz": TRAIN_IMGSZ, "epochs": TRAIN_EPOCHS,
              "finetune_epochs": FINETUNE_EPOCHS},
    "inference": {
        "conf": INFER_CONF, "imgsz": TRAIN_IMGSZ,
        "panel_create_conf": PANEL_CREATE_CONF,
        "panel_create_frames": PANEL_CREATE_FRAMES,
        "brand_color_snap_hue": BRAND_COLOR_SNAP_HUE,
        "propagation": {
            "max_propagation_frames": MAX_PROPAGATION_FRAMES,
            "pan_drop_px": PAN_DROP_PX,
            "verify_content": VERIFY_CONTENT,
            "content_hue_tol": CONTENT_HUE_TOL,
            "field_of_play": FIELD_OF_PLAY,
        },
        "merge_gap_height_ratio": MERGE_GAP_HEIGHT_RATIO,
        "panel_margin_frac": PANEL_MARGIN_FRAC,
        "panel_feather_px": PANEL_FEATHER_PX,
        "person_conf": PERSON_CONF,
        "person_hold_frames": PERSON_HOLD_FRAMES,
        "person_dilate_px": PERSON_DILATE_PX,
        "roi_person_pass": ROI_PERSON_PASS,
        "roi_upscale": ROI_UPSCALE,
        "max_board_height_frac": MAX_BOARD_HEIGHT_FRAC,
        "bottom_snap": {"enabled": BOTTOM_SNAP,
                        "max_shift_frac": BOTTOM_SNAP_MAX_FRAC},
        "occlusion": {
            "occlusion_zone_down_frac": OCCLUSION_ZONE_DOWN_FRAC,
            "occlusion_zone_up_px": OCCLUSION_ZONE_UP_PX,
            "feet_tolerance_frac": FEET_TOLERANCE_FRAC,
            "front_latch_frames": FRONT_LATCH_FRAMES,
        },
        "protect_goal": {"enabled": PROTECT_GOAL},
        "motion_person_fallback": MOTION_PERSON_FALLBACK,
        "mute": {
            "sample_band_frac": SAMPLE_BAND_FRAC,
            "saturation_mult": SATURATION_MULT,
            "value_mult": VALUE_MULT,
            "scene_blend": SCENE_BLEND,
            "max_relative_brightness": MAX_RELATIVE_BRIGHTNESS,
        },
    },
}
(REPO_ROOT / "config.local.yaml").write_text(
    "# GENERATED by notebooks/pipeline.ipynb - edit the notebook's config cell,\n"
    "# not this file. Delete it to fall back to config.yaml defaults.\n"
    + yaml.safe_dump(overlay, sort_keys=False)
)
cfg = load_config()
P = cfg["paths"]

def run(script, *args, title=None):
    """Run a pipeline script and stream its output."""
    cmd = [PY, str(REPO_ROOT / "scripts" / script), *[str(a) for a in args]]
    print(f"\n$ {' '.join(cmd[1:])}\n" + "-" * 70)
    r = subprocess.run(cmd, cwd=REPO_ROOT)
    if r.returncode != 0:
        raise RuntimeError(f"{script} failed (exit {r.returncode})")

def skip(step, why):
    print(f"[skip] {step}: {why}  (set FORCE_{step.upper()} = True to redo)")

def fingerprint(paths):
    """Stable hash of a file set - used to notice new/changed inputs."""
    h = hashlib.sha256()
    for p in sorted(paths):
        st = p.stat()
        h.update(f"{p.name}:{st.st_size}:{int(st.st_mtime)}".encode())
    return h.hexdigest()[:16]

def stamp_path(name):
    d = REPO_ROOT / "data" / ".stamps"
    d.mkdir(parents=True, exist_ok=True)
    return d / name

print(f"repo    : {REPO_ROOT}")
print(f"videos  : {len(find_videos(P['raw_videos']))} in input/videos/")
print(f"brands  : {sorted(d.name for d in P['logos'].iterdir() if d.is_dir())}")
print(f"config  : config.yaml + config.local.yaml (written above)")

---
## Step 0 — Reset (optional)

Deletes everything *generated*: frames, synthetic images, the assembled
dataset, trained weights, and outputs. Never touches `input/`.

In [ ]:
RESET = False   # ← set True, run this cell, then set it back to False

if RESET:
    import shutil
    for d in [P["frames"], P["synthetic"], P["dataset"], P["annotations_real"],
              P["negatives"], P["outputs"], REPO_ROOT / "data" / ".stamps",
              REPO_ROOT / "runs"]:
        if d.exists():
            shutil.rmtree(d)
            print(f"removed {d.relative_to(REPO_ROOT)}")
    for f in (P["models"] / "best.pt",):
        if f.exists():
            f.unlink()
            print(f"removed {f.relative_to(REPO_ROOT)}")
    print("\nReset complete - set RESET = False and Run All.")
else:
    print("RESET is False - nothing deleted.")

---
## Step 1 — Environment check

In [ ]:
import torch
print(f"python      : {sys.version.split()[0]}")
print(f"torch       : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()} "
      f"({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'})")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory  : {free/2**30:.1f} GB free / {total/2**30:.1f} GB total")
import shutil as _sh
print(f"ffmpeg      : {_sh.which('ffmpeg') or 'MISSING - install it before Step 2'}")

---
## Step 2 — Extract frames

Only videos that don't already have a frame folder are processed, so adding
one new video to `input/videos/` extracts just that one.

In [ ]:
videos = find_videos(P["raw_videos"])
if not videos:
    print("No videos found. Drop one into input/videos/ and re-run this cell.")

for v in videos:
    out_dir = P["frames"] / slugify(v.stem)
    have = out_dir.exists() and any(out_dir.iterdir())
    if have and not FORCE_EXTRACT:
        skip("extract", f"{v.name} already has {len(list(out_dir.iterdir()))} frames")
        continue
    args = ["--video", v.name, "--fps", EXTRACT_FPS]
    if FORCE_EXTRACT:
        args.append("--force")
    run("extract_frames.py", *args)

---
## Step 3 — Auto-label real betting boards

Finds the advertising strips in the extracted frames and writes real training
annotations, so the model learns from actual broadcast footage rather than
only from pasted logos. Frames already covered are skipped.

In [ ]:
lbl_dir = P["annotations_real"] / "labels"
have = lbl_dir.exists() and any(lbl_dir.glob("*.txt"))
if have and not FORCE_AUTOLABEL:
    skip("autolabel", f"{len(list(lbl_dir.glob('*.txt')))} real annotations already exist")
else:
    run("autolabel_betano.py")

---
## Step 4 — Generate synthetic training images

Re-runs only when the contents of `input/logos/` changed — so adding a new
brand folder regenerates, and a plain re-run doesn't.

In [ ]:
logo_files = [f for d in P["logos"].iterdir() if d.is_dir()
              for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTS]
fp = fingerprint(logo_files)
stamp = stamp_path("synthetic.logos")
unchanged = stamp.exists() and stamp.read_text().strip() == fp
have = (P["synthetic"] / "images" / "train").exists() and any((P["synthetic"] / "images" / "train").iterdir())

if have and unchanged and not FORCE_SYNTHETIC:
    skip("synthetic", f"{len(logo_files)} logo file(s) unchanged since last run")
else:
    if have and not unchanged:
        print(f"[changed] input/logos/ differs from last run - regenerating synthetic data")
    run("generate_synthetic.py")
    stamp.write_text(fp)

---
## Step 5 — Assemble the dataset

Merges real annotations + synthetic images + hard negatives into
`data/dataset/`. Always runs (it's cheap) and appends rather than wiping, so
previously split frames keep their train/val side.

In [ ]:
run("assemble_dataset.py", "--append")

---
## Step 6 — Auto-label people in the dataset

Marks people so training knows not to treat them as background. Skipped when
the dataset hasn't changed since the last pass.

In [ ]:
train_imgs = sorted((P["dataset"] / "images" / "train").glob("*"))
fp = fingerprint(train_imgs[:2000])
stamp = stamp_path("persons.dataset")
unchanged = stamp.exists() and stamp.read_text().strip() == fp

if unchanged and not FORCE_PERSONS:
    skip("persons", f"dataset unchanged ({len(train_imgs)} train images)")
else:
    run("autolabel_persons.py")
    stamp.write_text(fp)

---
## Step 7 — Train

Fine-tunes from `models/best.pt` when it exists (new data improves the current
model instead of starting over), otherwise trains fresh from COCO weights.

**⚠️ On confidence:** if real-video testing shows ~0.4–0.5 confidence on a
large, obvious board, the model is undertrained. The tracker in Step 9 keeps
hiding robust despite that, but it cannot manufacture recall. For production
quality you want 300+ annotated real frames covering all camera angles
**including motion-blurred pan frames**, then a full retrain — not a short
fine-tune.

In [ ]:
best = P["models"] / "best.pt"
if best.exists() and not FORCE_TRAIN:
    skip("train", f"{best.relative_to(REPO_ROOT)} exists")
    print("      (adding data? set FORCE_TRAIN = True to fine-tune it on the new material)")
elif best.exists():
    print(f"Fine-tuning from {best.name} for {FINETUNE_EPOCHS} epochs...")
    run("train.py", "--weights", best, "--epochs", FINETUNE_EPOCHS)
else:
    print(f"No checkpoint yet - training fresh for {TRAIN_EPOCHS} epochs...")
    run("train.py", "--epochs", TRAIN_EPOCHS)

---
## Step 8 — Sanity check

Confidence on a few held-out real frames. Numbers near zero mean the model
never learned the real board (retrain); ~0.4–0.5 means undertrained but
usable with the Step 9 tracker; 0.7+ is healthy.

In [ ]:
from ultralytics import YOLO
import numpy as np

model = YOLO(str(P["models"] / "best.pt"))
val_imgs = sorted((P["dataset"] / "images" / "val").glob("*"))
real = [p for p in val_imgs if not p.name.startswith(("synth_", "neg_"))][:6]

if not real:
    print("No real frames in the val split to check.")
for img in real:
    r = model.predict(str(img), conf=0.10, imgsz=TRAIN_IMGSZ, verbose=False)[0]
    board = [float(c) for c, k in zip(r.boxes.conf.tolist(), r.boxes.cls.tolist()) if int(k) == 1]
    print(f"{img.name[:52]:54s} boards={len(board)} best={max(board) if board else 0:.2f}")

---
## Step 9 — Process a video

Produces the hidden-ad output. With `DEBUG = True` it also writes a
side-by-side diagnostic video: **left** shows raw detections with confidence,
the person mask tinted with track ids, each panel's detection-union bound
(cyan) and the rendered panel (green = detected this frame, orange = carried
by camera motion through a detection gap); **right** is the final output.

In [ ]:
videos = find_videos(P["raw_videos"])
target = None
if PROCESS_VIDEO:
    target = next((v for v in videos if v.name == PROCESS_VIDEO), None)
    if target is None:
        raise FileNotFoundError(f"{PROCESS_VIDEO} not found in input/videos/")
elif videos:
    target = videos[0]

if target is None:
    print("No video to process.")
else:
    out = P["outputs"] / f"{target.stem}_blocked.mp4"
    if out.exists() and not FORCE_PROCESS:
        skip("process", f"{out.name} already exists")
    else:
        args = ["--input", target.name, "--output", out]
        if PROCESS_START is not None:
            args += ["--start", PROCESS_START]
        if PROCESS_END is not None:
            args += ["--end", PROCESS_END]
        if DEBUG:
            args.append("--debug")
        run("process_video.py", *args)
        print(f"\nOutput: {out}")
        if DEBUG:
            print(f"Debug : {out.with_name(out.stem + '_debug.mp4')}")

---
## Step 10 — Export for Android (optional)

Writes `models/best.tflite` (int8) — the file that ships inside the APK.

In [ ]:
EXPORT = False   # ← set True when you want the mobile model

if EXPORT:
    run("export_tflite.py")
else:
    print("EXPORT is False - skipping TFLite export.")